In [628]:
import pandas as pd
import numpy as np

In [629]:
df = pd.read_csv(
    "../data/raw/export-tranzactii-2022-sector-2.csv",
    sep=";"
)

df.shape

(4325, 10)

In [630]:
apartments = df[df["tip"] == "apartament"].copy()

apartments.shape

(3987, 10)

In [631]:
apartments = apartments.rename(columns={
    "tip": "property_type",
    "Tip strada": "street_type",
    "Denumire strada": "street_name",
    "Zona": "neighborhood",
    "S utila": "area_sqm",
    "S desfasurata": "built_area_sqm",
    "S teren": "land_area_sqm",
    "Pret tranzactie LEI": "price_lei",
    "an_constructie": "year_built",
    "Perioada tranzactie": "transaction_period"
})

apartments.columns

Index(['property_type', 'street_type', 'street_name', 'neighborhood',
       'area_sqm', 'built_area_sqm', 'land_area_sqm', 'price_lei',
       'year_built', 'transaction_period'],
      dtype='str')

In [632]:
apartments.head()

,property_type,street_type,street_name,neighborhood,area_sqm,built_area_sqm,land_area_sqm,price_lei,year_built,transaction_period
0,apartament,Alee,BISTRICIOARA,colentina,91.41,NaN,NaN,469908.0,1989.0,2022-T1
1,apartament,Alee,BISTRICIOARA,colentina,56.44,NaN,NaN,252241.0,1987.0,2022-T1
2,apartament,Alee,BRASOVENI,tei,52.40,NaN,NaN,373347.0,1985.0,2022-T1
3,apartament,Alee,BURIDAVA,colentina,43.56,NaN,NaN,316527.0,2016.0,2022-T1
4,apartament,Alee,CIRCULUI,tei,59.60,NaN,NaN,495000.0,1960.0,2022-T1


In [633]:
apartments["neighborhood"].value_counts(dropna=False)

neighborhood
centrala           841
pantelimon         717
tei                687
pipera sud         620
colentina          555
iancului           249
vatra luminoasa    117
floreasca           98
andronache          40
Colentina           27
ion creanga         19
-                   15
Tei                  2
Name: count, dtype: int64

In [634]:
apartments["neighborhood"] = (
    apartments["neighborhood"]
    .str.strip()
    .str.lower()
)

In [635]:
apartments["neighborhood"] = apartments["neighborhood"].replace("-", np.nan)

In [636]:
apartments["neighborhood"].value_counts(dropna=False)

neighborhood
centrala           841
pantelimon         717
tei                689
pipera sud         620
colentina          582
iancului           249
vatra luminoasa    117
floreasca           98
andronache          40
ion creanga         19
NaN                 15
Name: count, dtype: int64

In [637]:
apartments["year_built"].sort_values().tail(10)

1687    2022.0
2753    2022.0
1352    2022.0
1693    2022.0
1694    2022.0
1299    2022.0
4320    2022.0
4166    2087.0
2689       NaN
3748       NaN
Name: year_built, dtype: float64

In [638]:
apartments.loc[
    apartments["year_built"] > 2022,
    "year_built"
] = np.nan

In [639]:
apartments["year_built"].isna().sum()

np.int64(3)

In [640]:
apartments["year_built"].max()

np.float64(2022.0)

In [641]:
Q1 = apartments["area_sqm"].quantile(0.25)
Q3 = apartments["area_sqm"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

Q1, Q3, IQR, lower_bound, upper_bound

(np.float64(39.605),
 np.float64(65.305),
 np.float64(25.70000000000001),
 np.float64(1.0549999999999855),
 np.float64(103.85500000000002))

In [642]:
area_outliers = apartments[
    (apartments["area_sqm"] < lower_bound) |
    (apartments["area_sqm"] > upper_bound)
]

area_outliers.shape

(142, 10)

In [643]:
(apartments["area_sqm"] <= 0).sum()

np.int64(0)

In [644]:
apartments["area_sqm"].isna().sum()

np.int64(264)

In [645]:
apartments = apartments.dropna(subset=["area_sqm"])

apartments.shape

(3723, 10)

In [646]:
apartments["price_lei"].isna().sum()

np.int64(27)

In [647]:
(apartments["price_lei"] <= 0).sum()

np.int64(0)

In [648]:
apartments = apartments.dropna(subset=["price_lei"])

apartments.shape

(3696, 10)

In [649]:
apartments = apartments.drop(columns=[
    "property_type",
    "built_area_sqm",
    "land_area_sqm"
])

apartments.shape

(3696, 7)

In [650]:
apartments["price_lei"].describe()

count    3.696000e+03
mean     4.264641e+05
std      2.795574e+05
min      1.000000e+04
25%      2.574750e+05
50%      3.744615e+05
75%      5.005202e+05
max      3.388797e+06
Name: price_lei, dtype: float64

In [651]:
apartments.sort_values(
    "price_lei",
    ascending=False
)[
    ["neighborhood", "area_sqm", "price_lei", "year_built"]
].head(10)

,neighborhood,area_sqm,price_lei,year_built
3017,floreasca,197.93,3388797.0,2014.0
953,pipera sud,235.29,3311073.0,2006.0
2987,floreasca,217.00,3004308.0,1925.0
3015,floreasca,95.80,3001423.0,2020.0
3068,tei,53.04,2943994.0,1971.0
96,pipera sud,506.11,2719200.0,2022.0
698,pipera sud,161.60,2583000.0,2020.0
447,pantelimon,35.70,2411409.0,1995.0
3510,NaN,195.89,2100222.0,1899.0
2889,centrala,101.82,2071829.0,2020.0


In [652]:
apartments["price_per_sqm"] = (
    apartments["price_lei"] / apartments["area_sqm"]
)

apartments["price_per_sqm"].describe()

count      3696.000000
mean       8035.216417
std        7205.801042
min         312.500000
25%        5929.917167
50%        7191.738949
75%        8917.530597
max      166132.844575
Name: price_per_sqm, dtype: float64

In [653]:
apartments[
    (apartments["price_per_sqm"] > 30000) |
    (apartments["price_per_sqm"] < 2000)
][
    [
        "neighborhood",
        "area_sqm",
        "price_lei",
        "price_per_sqm",
        "year_built"
    ]
].sort_values(
    "price_per_sqm",
    ascending=False
)


,neighborhood,area_sqm,price_lei,price_per_sqm,year_built
1030,colentina,3.41,566513.0,166132.844575,2013.0
3550,centrala,3.40,545450.0,160426.470588,2020.0
1025,colentina,4.65,666242.0,143277.849462,2008.0
1028,colentina,3.34,457764.0,137055.089820,2000.0
4283,centrala,6.03,690000.0,114427.860697,1936.0
...,...,...,...,...,...
1625,colentina,48.12,25000.0,519.534497,1964.0
1360,pipera sud,200.43,101051.0,504.171032,2020.0
3598,colentina,72.63,35408.0,487.512047,1992.0
4305,colentina,82.21,37500.0,456.148887,2009.0


In [654]:
high_price_outliers = apartments[
    apartments["price_per_sqm"] > 30000
]

low_price_outliers = apartments[
    apartments["price_per_sqm"] < 2000
]

print("Extreme mari:", len(high_price_outliers))
print("Extreme mici:", len(low_price_outliers))

Extreme mari: 26
Extreme mici: 95


In [655]:
small_apartments = apartments[
    apartments["area_sqm"] < 15
]

print("Apartamente sub 15 m²:", len(small_apartments))

small_apartments[
    ["neighborhood", "area_sqm", "price_lei", "price_per_sqm"]
].sort_values("area_sqm")

Apartamente sub 15 m²: 59


,neighborhood,area_sqm,price_lei,price_per_sqm
1028,colentina,3.34,457764.0,137055.089820
3550,centrala,3.40,545450.0,160426.470588
1030,colentina,3.41,566513.0,166132.844575
3387,centrala,4.00,345342.0,86335.500000
637,centrala,4.55,42039.0,9239.340659
1025,colentina,4.65,666242.0,143277.849462
1335,pipera sud,4.75,384948.0,81041.684211
1336,pipera sud,4.76,365606.0,76807.983193
289,centrala,5.25,556256.0,105953.523810
2412,centrala,5.40,514030.0,95190.740741


In [656]:
small_apartments["area_sqm"].describe()

count    59.000000
mean     11.608814
std       3.935679
min       3.340000
25%       9.020000
50%      13.770000
75%      14.545000
max      14.940000
Name: area_sqm, dtype: float64

In [657]:
print("Sub 10 m²:", (apartments["area_sqm"] < 10).sum())

print("Între 10 și 15 m²:",
      apartments["area_sqm"].between(10, 15, inclusive="left").sum())

Sub 10 m²: 17
Între 10 și 15 m²: 42


In [658]:
pd.crosstab(
    apartments["area_sqm"] < 10,
    apartments["price_per_sqm"] > 30000,
    rownames=["Area under 10 sqm"],
    colnames=["Price over 30000 lei/sqm"]
)

Price over 30000 lei/sqm,False,True
Area under 10 sqm,,
False,3667,12
True,3,14


In [659]:
apartments_before_area_filter = apartments.copy()

apartments = apartments[
    apartments["area_sqm"] >= 10
].copy()

apartments.shape

(3679, 8)

In [660]:
apartments[
    apartments["price_per_sqm"] < 2000
][
    [
        "neighborhood",
        "area_sqm",
        "price_lei",
        "price_per_sqm",
        "year_built"
    ]
].sort_values("price_per_sqm").head(20)

,neighborhood,area_sqm,price_lei,price_per_sqm,year_built
4047,centrala,32.00,10000.0,312.500000,1981.0
4305,colentina,82.21,37500.0,456.148887,2009.0
3598,colentina,72.63,35408.0,487.512047,1992.0
1360,pipera sud,200.43,101051.0,504.171032,2020.0
1625,colentina,48.12,25000.0,519.534497,1964.0
2289,centrala,80.81,44541.0,551.181784,1980.0
1940,pantelimon,80.80,45000.0,556.930693,1977.0
3187,tei,32.60,18701.0,573.650307,1975.0
531,colentina,63.90,37772.0,591.111111,1976.0
1421,pipera sud,150.00,89561.0,597.073333,2020.0


In [661]:
low_prices = apartments[
    apartments["price_per_sqm"] < 2000
]

low_prices["price_per_sqm"].describe()

count      95.000000
mean     1288.951603
std       510.739663
min       312.500000
25%       756.657805
50%      1363.087201
75%      1775.225048
max      1995.836162
Name: price_per_sqm, dtype: float64

In [662]:
apartments_before_price_filter = apartments.copy()

apartments = apartments[
    apartments["price_per_sqm"] >= 2000
].copy()

apartments.shape

(3584, 8)

In [663]:
apartments.isna().sum()

street_type            0
street_name            0
neighborhood          12
area_sqm               0
price_lei              0
year_built             3
transaction_period     0
price_per_sqm          0
dtype: int64

In [664]:
apartments["neighborhood"] = apartments["neighborhood"].fillna("unknown")

In [665]:
apartments["neighborhood"] = apartments["neighborhood"].fillna("unknown")

apartments.isna().sum()

street_type           0
street_name           0
neighborhood          0
area_sqm              0
price_lei             0
year_built            3
transaction_period    0
price_per_sqm         0
dtype: int64

In [666]:
print("STREET TYPES:")
print(apartments["street_type"].value_counts())

print("\nTRANSACTION PERIODS:")
print(apartments["transaction_period"].value_counts())

STREET TYPES:
street_type
Strada        1746
Sosea         1195
Bulevard       272
Alee           258
Calea           72
Intrare         35
Piata            3
Prelungire       3
Name: count, dtype: int64

TRANSACTION PERIODS:
transaction_period
2022-T3    949
2022-T2    924
2022-T1    878
2022-T4    833
Name: count, dtype: int64


In [667]:
apartments["transaction_quarter"] = (
    apartments["transaction_period"]
    .str.extract(r"T([1-4])")
    .astype(int)
)

apartments[
    ["transaction_period", "transaction_quarter"]
].head(10)

,transaction_period,transaction_quarter
0,2022-T1,1
1,2022-T1,1
2,2022-T1,1
3,2022-T1,1
4,2022-T1,1
5,2022-T1,1
6,2022-T1,1
7,2022-T1,1
9,2022-T1,1
10,2022-T1,1


In [668]:
apartments["transaction_quarter"].value_counts().sort_index()

transaction_quarter
1    878
2    924
3    949
4    833
Name: count, dtype: int64

In [669]:
print("Dataset shape:", apartments.shape)

print("Duplicate rows:", apartments.duplicated().sum())

print("\nMissing values:")
print(apartments.isna().sum())

Dataset shape: (3584, 9)
Duplicate rows: 6

Missing values:
street_type            0
street_name            0
neighborhood           0
area_sqm               0
price_lei              0
year_built             3
transaction_period     0
price_per_sqm          0
transaction_quarter    0
dtype: int64


In [670]:
apartments.to_csv(
    "../data/processed/clean_apartments.csv",
    index=False
)